# Laboratorium 1: od problemu biznesowego do projektu Data Mining

**Przypadek:** NovaTel — fikcyjny operator usług abonamentowych.  
**Cel:** zrozumieć problem biznesowy, rozpoznać dane i ich jakość oraz zaplanować projekt według CRISP-DM.

> Na tym laboratorium nie budujemy modelu. Model bez poprawnie zdefiniowanego problemu jest zwykle bardzo sprawnym sposobem uzyskania precyzyjnej odpowiedzi na niewłaściwe pytanie.

## Zasady pracy

1. Każdy wniosek powinien wynikać z danych albo z opisu sytuacji biznesowej.
2. Nie poprawiajcie jeszcze danych — najpierw je zdiagnozujcie.

In [86]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
DATA_PATH = Path("dane_novatel_lab1.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/lab1_data_mining_materialy/dane_novatel_lab1.csv")

df = pd.read_csv(DATA_PATH)

## Zadanie 1. Pierwsze spojrzenie na dane

Sprawdź wymiary zbioru, pięć pierwszych rekordów i nazwy kolumn.

In [87]:
print(f"Wymiary: {df.shape[0]} wierszy, {df.shape[1]} kolumn")
print("\nPięć pierwszych rekordów:")
display(df.head(5))
print("\nNazwy kolumn:")
print(list(df.columns))

Wymiary: 325 wierszy, 21 kolumn

Pięć pierwszych rekordów:


,customer_id,snapshot_date,wiek,płeć,województwo,wielkość_miejscowości,typ_umowy,staż_miesiące,opłata_miesięczna_pln,średnie_użycie_h_mies,...,kontakty_z_obsługą_3m,reklamacje_6m,satysfakcja_1_5,zgoda_marketingowa,kanał_pozyskania,rabat_proc,dni_od_ostatniego_logowania,dochód_miesięczny_pln,odejście_90d,data_zamknięcia_konta
0,NT10234,2026-10-01,38.0,M,Mazowieckie,wieś,miesięczna,9,96.23,32.3,...,1,0,4.0,tak,online,24.0,3,13700.0,tak,2026-12-04
1,NT10110,2026-10-01,45.0,inna/nie podano,Mazowieckie,średnie miasto,12 miesięcy,67,100.62,63.9,...,1,3,4.0,nie,salon,13.0,9,18840.0,nie,NaN
2,NT10248,2026-10-01,36.0,M,Pomorskie,małe miasto,12 miesięcy,16,115.73,28.7,...,2,0,3.0,tak,salon,7.0,6,4220.0,nie,NaN
3,NT10009,2026-10-01,34.0,M,Dolnośląskie,średnie miasto,12 miesięcy,10,110.10,39.0,...,1,0,3.0,nie,telefon,4.0,4,9020.0,nie,NaN
4,NT10093,2026-10-01,34.0,M,Mazowieckie,duże miasto,miesięczna,32,117.50,38.8,...,1,0,4.0,nie,online,15.0,5,10260.0,tak,2026-12-26



Nazwy kolumn:
['customer_id', 'snapshot_date', 'wiek', 'płeć', 'województwo', 'wielkość_miejscowości', 'typ_umowy', 'staż_miesiące', 'opłata_miesięczna_pln', 'średnie_użycie_h_mies', 'opóźnienia_płatności_6m', 'kontakty_z_obsługą_3m', 'reklamacje_6m', 'satysfakcja_1_5', 'zgoda_marketingowa', 'kanał_pozyskania', 'rabat_proc', 'dni_od_ostatniego_logowania', 'dochód_miesięczny_pln', 'odejście_90d', 'data_zamknięcia_konta']


**Do karty pracy:**  
- Ile jest obserwacji i zmiennych?  - Zbiór ma 325 obserwacji (wierszy) i 21 zmiennych (kolumn).
- Co jest jednostką obserwacji? - Jednostką obserwacji jest pojedynczy klient w dniu migawki danych. Zbiór zawiera 325 rekordów dotyczących 320 unikalnych klientów.
- Czy każda kolumna jest potencjalnie przydatna przed podjęciem decyzji? - Nie. customer_id to tylko techniczny identyfikator. odejście_90d to zmienna celu, więc nie może być cechą. data_zamknięcia_konta jest wypełniona tylko u klientów, którzy odeszli, i jest późniejsza niż data migawki, więc to wyciek danych. snapshot_date ma tę samą wartość u wszystkich klientów, więc niczego nie różnicuje. Kolumny wrażliwe (wiek, płeć, województwo) mogą rodzić ryzyko dyskryminacji przy podejmowaniu decyzji, więc trzeba je stosować ostrożnie.

## Zadanie 2. Typy danych i zmienna celu

In [88]:
df.info()
counts = df['odejście_90d'].value_counts(dropna=False)
share = df['odejście_90d'].value_counts(dropna=False, normalize=True).mul(100).round(1)
rozklad = pd.DataFrame({"liczba klientów": counts, "udział [%]": share})
print("Rozkład zmiennej celu (odejście_90d):")
display(rozklad)

<class 'pandas.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   customer_id                  325 non-null    str    
 1   snapshot_date                325 non-null    str    
 2   wiek                         313 non-null    float64
 3   płeć                         325 non-null    str    
 4   województwo                  325 non-null    str    
 5   wielkość_miejscowości        325 non-null    str    
 6   typ_umowy                    325 non-null    str    
 7   staż_miesiące                325 non-null    int64  
 8   opłata_miesięczna_pln        325 non-null    float64
 9   średnie_użycie_h_mies        325 non-null    float64
 10  opóźnienia_płatności_6m      325 non-null    int64  
 11  kontakty_z_obsługą_3m        325 non-null    int64  
 12  reklamacje_6m                325 non-null    int64  
 13  satysfakcja_1_5              30

,liczba klientów,udział [%]
odejście_90d,,
nie,296,91.1
tak,29,8.9


**Do karty pracy:**  
- Jaki jest udział klientów, którzy odeszli? - W surowym zbiorze 29 z 325 rekordów ma wartość odejście_90d = tak, czyli 8,9%. Pozostałe 296 rekordów, czyli 91,1%, ma wartość „nie”.
- Czy klasy są zrównoważone?  - Nie. Klasa tak (odejście) jest mniejszościowa i stanowi niecałe 9% obserwacji.
- Dlaczego sama accuracy mogłaby później wprowadzać w błąd? - Model, który dla każdego klienta przewidywałby nie, miałby accuracy około 91,1%, mimo że nie wykryłby ani jednego odchodzącego klienta. Wysoka accuracy wynika więc głównie z przewagi klasy większościowej i nie mówi nic o tym, czy model rozpoznaje klientów zagrożonych odejściem.

## Zadanie 3. Braki danych

In [89]:
missing = df.isna().sum().sort_values(ascending=False)
braki = missing[missing > 0].to_frame("Liczba braków")
braki["Udział [%]"] = (braki["Liczba braków"] / len(df) * 100).round(1)
display(braki)

,Liczba braków,Udział [%]
data_zamknięcia_konta,296,91.1
dochód_miesięczny_pln,28,8.6
satysfakcja_1_5,19,5.8
wiek,12,3.7
kanał_pozyskania,7,2.2


**Do karty pracy:**  
Wskaż kolumny z brakami i zaproponuj pytania, które należałoby zadać właścicielowi danych przed ich uzupełnianiem lub usuwaniem.

Kolumny z brakami: data_zamknięcia_konta (296), dochód_miesięczny_pln (28), satysfakcja_1_5 (19), wiek (12), kanał_pozyskania (7).

Pytania do właściciela danych:
- Czy puste pole w data_zamknięcia_konta` oznacza, że klient nie zamknął konta, i czy ta data jest znana już w dniu snapshot_date?
- Czy dochód_miesięczny_pln jest podawany dobrowolnie i czy jego brak wynika z odmowy klienta, czy z błędu zbierania danych?
- Czy brak w satysfakcja_1_5 oznacza, że klient nie brał udziału w badaniu, i jak dobierano klientów do ankiety?
- Czy wiek jest obowiązkowy przy zawarciu umowy i czy jego brak wynika z błędu migracji danych z CRM?
- Czy brak w kanał_pozyskania to błąd rejestracji w CRM, czy kanał nie był wtedy odnotowywany?

## Zadanie 4. Duplikaty i identyfikatory

In [90]:
print("Liczba duplikatów: ", df.duplicated().sum())
print("Liczba zduplikowanych id: ", df['customer_id'].duplicated().sum())
print("Liczba unikalnych klientów: ", df['customer_id'].nunique())
display(df[df['customer_id'].duplicated(keep=False)].sort_values('customer_id'))

Liczba duplikatów:  5
Liczba zduplikowanych id:  5
Liczba unikalnych klientów:  320


,customer_id,snapshot_date,wiek,płeć,województwo,wielkość_miejscowości,typ_umowy,staż_miesiące,opłata_miesięczna_pln,średnie_użycie_h_mies,...,kontakty_z_obsługą_3m,reklamacje_6m,satysfakcja_1_5,zgoda_marketingowa,kanał_pozyskania,rabat_proc,dni_od_ostatniego_logowania,dochód_miesięczny_pln,odejście_90d,data_zamknięcia_konta
22,NT10046,2026-10-01,38.0,K,Dolnośląskie,średnie miasto,miesięczna,34,51.95,10.7,...,1,2,2.0,nie,salon,4.0,2,9040.0,nie,NaN
60,NT10046,2026-10-01,38.0,K,Dolnośląskie,średnie miasto,miesięczna,34,51.95,10.7,...,1,2,2.0,nie,salon,4.0,2,9040.0,nie,NaN
192,NT10142,2026-10-01,32.0,K,Wielkopolskie,duże miasto,miesięczna,39,57.25,45.3,...,1,0,NaN,tak,online,20.0,4,5960.0,nie,NaN
318,NT10142,2026-10-01,32.0,K,Wielkopolskie,duże miasto,miesięczna,39,57.25,45.3,...,1,0,NaN,tak,online,20.0,4,5960.0,nie,NaN
14,NT10191,2026-10-01,64.0,M,Wielkopolskie,średnie miasto,12 miesięcy,26,39.18,16.7,...,3,0,3.0,tak,partner,14.0,2,9490.0,nie,NaN
307,NT10191,2026-10-01,64.0,M,Wielkopolskie,średnie miasto,12 miesięcy,26,39.18,16.7,...,3,0,3.0,tak,partner,14.0,2,9490.0,nie,NaN
228,NT10216,2026-10-01,50.0,K,Dolnośląskie,duże miasto,24 miesiące,41,92.65,48.0,...,2,0,5.0,tak,telefon,13.0,23,4630.0,nie,NaN
259,NT10216,2026-10-01,50.0,K,Dolnośląskie,duże miasto,24 miesiące,41,92.65,48.0,...,2,0,5.0,tak,telefon,13.0,23,4630.0,nie,NaN
0,NT10234,2026-10-01,38.0,M,Mazowieckie,wieś,miesięczna,9,96.23,32.3,...,1,0,4.0,tak,online,24.0,3,13700.0,tak,2026-12-04
297,NT10234,2026-10-01,38.0,M,Mazowieckie,wieś,miesięczna,9,96.23,32.3,...,1,0,4.0,tak,online,24.0,3,13700.0,tak,2026-12-04


## Zadanie 5. Kategorie niespójne

In [91]:
for col in ["województwo", "kanał_pozyskania", "typ_umowy"]:
    print("\n", col)
    print(df[col].value_counts(dropna=False).rename(index=repr))


 województwo
województwo
'Mazowieckie'      78
'Śląskie'          62
'Dolnośląskie'     57
'Wielkopolskie'    48
'Małopolskie'      37
'Pomorskie'        34
'Pomorskie '        3
'MAZOWIECKIE'       3
'dolnośląskie'      3
Name: count, dtype: int64

 kanał_pozyskania
kanał_pozyskania
'online'      150
'salon'        68
'telefon'      50
'partner'      42
nan             7
'Online'        3
'telefon '      3
'SALON'         2
Name: count, dtype: int64

 typ_umowy
typ_umowy
'miesięczna'     154
'12 miesięcy'    106
'24 miesiące'     65
Name: count, dtype: int64


## Zadanie 6. Wartości podejrzane

In [92]:
display(df[["wiek", "staż_miesiące", "opłata_miesięczna_pln", "satysfakcja_1_5"]].describe())
invalid = pd.concat([
    df[(df["wiek"] < 18) | (df["wiek"] > 100)].assign(problem="wiek poza zakresem"),
    df[df["opłata_miesięczna_pln"] < 0].assign(problem="opłata ujemna"),
    df[(df["satysfakcja_1_5"] < 1) | (df["satysfakcja_1_5"] > 5)].assign(problem="satysfakcja poza skalą 1-5"),
    df[df["staż_miesiące"] <= 0].assign(problem="staż <= 0")])

print("Liczba podejrzanych rekordów:\n", invalid["problem"].value_counts())
display(invalid[["customer_id", "wiek", "staż_miesiące", "opłata_miesięczna_pln", "satysfakcja_1_5", "problem"]])

,wiek,staż_miesiące,opłata_miesięczna_pln,satysfakcja_1_5
count,313.000000,325.000000,325.000000,306.000000
mean,40.798722,27.966154,95.652985,3.673203
std,13.187543,17.492558,28.663192,0.977317
min,18.000000,0.000000,-49.990000,1.000000
25%,32.000000,15.000000,76.860000,3.000000
50%,40.000000,25.000000,95.080000,4.000000
75%,48.000000,39.000000,115.100000,4.000000
max,129.000000,88.000000,175.860000,7.000000


Liczba podejrzanych rekordów:
 problem
wiek poza zakresem            1
opłata ujemna                 1
satysfakcja poza skalą 1-5    1
staż <= 0                     1
Name: count, dtype: int64


,customer_id,wiek,staż_miesiące,opłata_miesięczna_pln,satysfakcja_1_5,problem
16,NT10146,129.0,3,100.81,5.0,wiek poza zakresem
202,NT10089,47.0,39,-49.99,4.0,opłata ujemna
264,NT10296,18.0,6,145.11,7.0,satysfakcja poza skalą 1-5
135,NT10260,47.0,0,124.38,4.0,staż <= 0


## Zadanie 7. Prosta eksploracja biznesowa

In [93]:
cechy = ["staż_miesiące", "opłata_miesięczna_pln", "satysfakcja_1_5",
         "kontakty_z_obsługą_3m", "dni_od_ostatniego_logowania"]
print("Średnie wartości cech: klienci, którzy odeszli (tak) vs pozostali (nie):")
display(df.groupby("odejście_90d")[cechy].mean().round(1))
print("\nOdsetek klientów, którzy odeszli [%], według typu umowy:")
display(df.groupby("typ_umowy")["odejście_90d"].apply(lambda s: round((s == "tak").mean() * 100, 1)))
print("\nLiczba klientów i odejść według typu umowy:")
display(df.groupby("typ_umowy")["odejście_90d"].agg(klienci="count", odeszło=lambda s: (s == "tak").sum()))
print("\nŚredni rabat [%] według odejścia:")
display(df.groupby("odejście_90d")["rabat_proc"].mean().round(1))

Średnie wartości cech: klienci, którzy odeszli (tak) vs pozostali (nie):


,staż_miesiące,opłata_miesięczna_pln,satysfakcja_1_5,kontakty_z_obsługą_3m,dni_od_ostatniego_logowania
odejście_90d,,,,,
nie,28.0,95.1,3.7,1.3,8.5
tak,28.1,101.4,3.6,1.2,7.9



Odsetek klientów, którzy odeszli [%], według typu umowy:


typ_umowy
12 miesięcy     9.4
24 miesiące     1.5
miesięczna     11.7
Name: odejście_90d, dtype: float64


Liczba klientów i odejść według typu umowy:


,klienci,odeszło
typ_umowy,,
12 miesięcy,106,10
24 miesiące,65,1
miesięczna,154,18



Średni rabat [%] według odejścia:


odejście_90d
nie     9.9
tak    12.0
Name: rabat_proc, dtype: float64

**Do karty pracy:**  
Zapisz maksymalnie trzy obserwacje. To mają być hipotezy lub sygnały do dalszej analizy, a nie ostateczne dowody przyczynowości.

- Typ umowy: odsetek odejść jest najniższy przy umowie na 24 miesiące (1 z 65 rekordów, 1,5%), a wyższy przy 12-miesięcznej (10 ze 106, 9,4%) i miesięcznej (18 ze 154, 11,7%). Hipoteza: dłuższe zobowiązanie zmniejsza skłonność do odejścia. Wynik dla 24 miesięcy opiera się na jednym odejściu, więc wymaga ostrożności.
- Opłata i rabat: klienci, którzy odeszli, płacą średnio więcej (101,4 vs 95,1 PLN), mimo że mają też wyższy średni rabat (12,0% vs 9,9%). Hipoteza: wyższa cena zwiększa ryzyko odejścia, a rabat nie tłumaczy tej różnicy. Trzeba sprawdzić, czy opłata jest podana przed rabatem, czy po nim.
- Pozostałe cechy: staż, satysfakcja, kontakty z obsługą i dni od logowania mają podobne średnie w obu grupach. Hipoteza: same średnie nie wystarczą, trzeba badać rozkłady i zależności między cechami.

## Zadanie 8. CRISP-DM

Uzupełnij w karcie pracy plan projektu dla sześciu etapów:

1. Business Understanding  
2. Data Understanding  
3. Data Preparation  
4. Modeling  
5. Evaluation  
6. Deployment

Dla każdego etapu podaj:
- najważniejsze pytanie,
- wymagany rezultat,
- ryzyko lub warunek powodzenia.

Plan projektu (CRISP-DM)
| Etap | Najważniejsze pytanie | Rezultat etapu | Ryzyko / warunek powodzenia |
|---|---|---|---|
| Business Understanding | Jak wykorzystać przewidywanie odejść do ograniczenia rezygnacji klientów? | Uzgodniony cel biznesowy, definicja odejścia, sposób działania i miernik sukcesu kampanii. | Brak konkretnej decyzji biznesowej lub nieuwzględnienie kosztów działań. |
| Data Understanding | Czy dane są odpowiednie do przewidywania odejścia w ciągu 90 dni? | Opis zmiennych, rozkład celu oraz lista braków, duplikatów, niespójności i podejrzanych wartości. | Niewielka liczba odejść, problemy jakości danych i niepewna dostępność cech w momencie predykcji. |
| Data Preparation | Jak przygotować dane bez utraty ważnych informacji i wycieku danych? | Zbiór z obsłużonymi duplikatami, kategoriami, brakami i błędami; wykluczenie daty zamknięcia z cech modelu. | Poprawki wymagają uzasadnienia; parametry uzupełniania braków i skalowania należy wyznaczać wyłącznie na danych treningowych. |
| Modeling | Który model najlepiej rozpoznaje klientów zagrożonych odejściem? | Porównanie prostych modeli, np. regresji logistycznej i drzewa, z modelem bazowym przewidującym brak odejścia. | Przeuczenie i niestabilne wyniki z powodu małej liczby odejść; ten sam klient nie może trafić do treningu i testu. |
| Evaluation | Czy model wykrywa odejścia i pomaga podejmować opłacalne decyzje? | Ocena na danych niewykorzystanych do uczenia: recall, precyzja, F1, macierz pomyłek oraz analiza kosztów przy wybranym progu ryzyka. | Sama accuracy może wprowadzać w błąd; przy niewielu odejściach ocena będzie obarczona dużą niepewnością. |
| Deployment | Jak włączyć wyniki do pracy zespołu utrzymania klienta? | Lista klientów do kontaktu, zasady wykorzystania wyniku oraz plan monitorowania modelu i kampanii. | Zmiany zachowań klientów, ograniczone zasoby zespołu i ryzyko niewłaściwego wykorzystania wyniku. |

## Zadanie 9. Etyka, prywatność i data leakage

In [94]:
print("Odejście vs czy podano datę zamknięcia konta:")
display(pd.crosstab(df["odejście_90d"], df["data_zamknięcia_konta"].notna(),
                    colnames=["data zamknięcia podana"]))
dni = (pd.to_datetime(df["data_zamknięcia_konta"]) - pd.to_datetime(df["snapshot_date"])).dt.days
print("\nDni od snapshot_date do zamknięcia konta:")
display(dni.describe())

Odejście vs czy podano datę zamknięcia konta:


data zamknięcia podana,False,True
odejście_90d,,
nie,296,0
tak,0,29



Dni od snapshot_date do zamknięcia konta:


count    29.000000
mean     43.724138
std      28.836852
min       4.000000
25%      13.000000
50%      45.000000
75%      72.000000
max      86.000000
dtype: float64

Odpowiedz:

- Czy `data_zamknięcia_konta` może być używana do przewidywania odejścia przed jego wystąpieniem? - Nie. Kolumna jest wypełniona we wszystkich 29 rekordach z odejściem i w żadnym z 296 rekordów bez odejścia, więc sama obecność daty idealnie odtwarza zmienną celu. Data przypada od 4 do 86 dni po snapshot_date, czyli w momencie predykcji jeszcze jej nie znamy. To data leakage: model miałby sztucznie wysoką skuteczność, a w praktyce byłby bezużyteczny. Kolumnę trzeba usunąć z cech.
- Jakie ryzyka wiążą się z wykorzystaniem wieku, płci, miejsca zamieszkania i dochodu? - Ryzyko dyskryminacji klientów ze względu na cechy osobiste, odtwarzania uprzedzeń zawartych w danych, naruszenia prywatności i przepisów o ochronie danych (RODO). Usunięcie tych cech nie zawsze wystarcza, bo inne zmienne mogą być z nimi skorelowane (np. województwo i dochód).
- Czy NovaTel powinien automatycznie odmawiać ofert lub podnosić ceny na podstawie modelu? - Nie. Model daje tylko prawdopodobieństwo i może się mylić, a odmowa oferty lub podwyżka ceny krzywdzi klienta (także tego, który by nie odszedł). Model powinien wspierać działania utrzymaniowe, np. ofertę dla zagrożonych klientów, a decyzje o cenach powinien podejmować człowiek.
- Jakie dane naprawdę są konieczne do rozwiązania problemu? - W pierwszej kolejności należy rozważyć dane o korzystaniu z usługi i relacji z klientem. Konieczność użycia wieku, płci, miejsca zamieszkania i dochodu wymaga osobnego uzasadnienia. customer_id służy do identyfikacji klienta, ale nie powinien być cechą predykcyjną. data_zamknięcia_konta należy wykluczyć z cech z powodu wycieku informacji.

**Rekomendacja dla NovaTel:** Przed budową modelu należy zweryfikować i uporządkować dane oraz wykluczyć z cech predykcyjnych data_zamknięcia_konta, a następnie ocenić model pod kątem wykrywania odejść i opłacalności działań utrzymaniowych, wykorzystując jego wyniki do wspierania decyzji pracowników.

## Oddanie

Oddaj:
- uzupełnioną kartę pracy, 
- notebook z wykonanymi analizami, 
- jednozdaniową rekomendację dalszego działania dla NovaTel. 